# Week Exercise - week 8

Multi-Agent Pricer Implementation

In [ ]:
# Import numpy first so PyTorch (loaded by gradio/sentence_transformers) can use it
import numpy as np
import os
import json
import re
from openai import OpenAI
import chromadb
import gradio as gr
import logging
from dotenv import load_dotenv
from agents.messaging_agent import MessagingAgent
from agents.deals import Deal, Opportunity, DealSelection
from deal_agent_framework import DealAgentFramework
from agents.scanner_agent import ScannerAgent
from agents.frontier_agent import FrontierAgent

In [2]:
from agents.agent import Agent

In [3]:
load_dotenv(override=True)
logging.getLogger().setLevel(logging.INFO)

In [ ]:
# LOAD API KEYS
openai_api_key = os.getenv('OPENAI_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openrouter_base_url = os.getenv('OPENROUTER_BASE_URL')

if openai_api_key:
    print(f"OpenAI API Key exists")
else:
    print("OpenAI API Key not set")

if openrouter_api_key:
    print(f"OpenRouter API Key exists")
else:
    print("OpenRouter API Key not set")

In [5]:
#initialize agents

openai = OpenAI()
openrouter_client = OpenAI(base_url=openrouter_base_url, api_key=openrouter_api_key)

In [6]:
def get_price(s: str) -> float:
    s = (s or "").replace("$", "").replace(",", "")
    m = re.search(r"[-+]?\d*\.\d+|\d+", s)
    return float(m.group()) if m else 0.0

def _chat_price(client: OpenAI, model: str, description: str) -> float:
    r = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": f"Estimate the price in USD. Reply with only the number.\n\n{description}"}],
        max_tokens=32,
    )
    return get_price(r.choices[0].message.content or "")

In [7]:
class OpenRouterAgent(Agent):
    name, color = "OpenRouter", Agent.CYAN
    def __init__(self, model_id: str):
        self.client = openrouter_client
        self.model = model_id
    def price(self, description: str) -> float:
        return _chat_price(self.client, self.model, description)

In [8]:
DB = "products_vectorstore_db"
client = chromadb.PersistentClient(path=DB)
collection = client.get_or_create_collection("products")

In [ ]:
openai = FrontierAgent(collection)
claude_3_5_sonnet_client = OpenRouterAgent("anthropic/claude-3.5-sonnet")
gpt_oss_client = OpenRouterAgent("openai/gpt-oss-120b")
gemini_1_5_pro_client = OpenRouterAgent("google/gemini-1.5-pro")

weights = (0.5, 0.2, 0.2, 0.1)

In [32]:
def ensemble_price(description: str) -> float:
    model_clients = [
        (weights[0], openai, "FrontierAgent"),
        (weights[1], claude_3_5_sonnet_client, "Claude 3.5 Sonnet"),
        (weights[2], gpt_oss_client, "GPT-OSS 120B"),
        (weights[3], gemini_1_5_pro_client, "Gemini 1.5 Pro"),
    ]
    
    prices = []
    active_weights = []

    for w, client, name in model_clients:
        try:
            price = client.price(description)
            prices.append(price)
            active_weights.append(w)
        except Exception as e:
            logging.warning("Skipping %s (failed): %s", name, e)

    if not prices:
        return 0.0  # All models failed
    
    # Normalize the weights to sum to 1
    total_weight = sum(active_weights)
    normalized_weights = [w / total_weight for w in active_weights]

    # Compute weighted average
    ensemble = sum(p * w for p, w in zip(prices, normalized_weights))
    return ensemble

In [33]:
class PlanningAgentLLM(Agent):
    name, color, DEAL_THRESHOLD = "Planning (LLM)", Agent.GREEN, 50
    def __init__(self, collection):
        self.scanner = ScannerAgent()
        self.messenger = MessagingAgent()
        self._collection = collection
    def run(self, deal: Deal) -> Opportunity:
        est = ensemble_price(deal.product_description)
        return Opportunity(deal=deal, estimate=est, discount=est - deal.price)
    def plan(self, memory=None):
        memory = memory or []
        sel = self.scanner.scan(memory=memory)
        if not sel: return None
        opps = [self.run(d) for d in sel.deals[:5]]
        opps.sort(key=lambda o: o.discount, reverse=True)
        best = opps[0]
        if best.discount > self.DEAL_THRESHOLD:
            try:
                self.messenger.alert(best)
            except Exception as e:
                logging.warning("Messenger alert skipped: %s", e)
        return best if best.discount > self.DEAL_THRESHOLD else None

In [ ]:
%pip install "numpy<2"

In [ ]:
framework = DealAgentFramework()
framework.planner = PlanningAgentLLM(framework.collection)
opportunities = framework.run()
print(len(opportunities), opportunities[0] if opportunities else None)

In [37]:

class AutonomousPlannerLLM(Agent):
    name, color = "Autonomous (LLM)", Agent.GREEN

    def __init__(self):
        self.scanner = ScannerAgent()
        self.messenger = MessagingAgent()
        self.openai = OpenAI()
        self.memory = None
        self.opportunity = None

    # Safe scan
    def scan_(self):
        try:
            r = self.scanner.scan(memory=self.memory)
            return r.model_dump_json() if r else "No deals"
        except Exception as e:
            logging.warning(f"[Scan failed] {e}")
            return "Scan failed"

    # Safe estimation using ensemble_price
    def estimate_(self, description: str):
        try:
            return str(ensemble_price(description))
        except Exception as e:
            logging.warning(f"[Estimate failed] {description} -> {e}")
            return "0.0"

    # Safe notification
    def notify_(self, description: str, deal_price: float, est: float, url: str):
        try:
            self.messenger.notify(description, deal_price, float(est), url)
            self.opportunity = Opportunity(
                deal=Deal(product_description=description, price=deal_price, url=url),
                estimate=float(est),
                discount=float(est) - deal_price
            )
            return "ok"
        except Exception as e:
            logging.warning(f"[Notify failed] {description} -> {e}")
            return "notify failed"

    # Tool definitions
    def get_tools(self):
        return [
            {"type": "function", "function": {"name": "scan", "description": "Get bargains", "parameters": {"type": "object", "properties": {}}}},
            {"type": "function", "function": {"name": "estimate", "description": "Estimate value", "parameters": {"type": "object", "properties": {"description": {"type": "string"}}, "required": ["description"]}}},
            {"type": "function", "function": {"name": "notify", "description": "Notify user of deal", "parameters": {"type": "object", "properties": {"description": {"type": "string"}, "deal_price": {"type": "number"}, "estimated_true_value": {"type": "number"}, "url": {"type": "string"}}, "required": ["description", "deal_price", "estimated_true_value", "url"]}}},
        ]

    # Execute tool calls safely
    def handle_tool(self, msg):
        out = []
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            try:
                if name == "scan":
                    res = self.scan_()
                elif name == "estimate":
                    res = self.estimate_(args.get("description", ""))
                elif name == "notify":
                    res = self.notify_(
                        args.get("description", ""),
                        args.get("deal_price", 0),
                        args.get("estimated_true_value", 0),
                        args.get("url", "")
                    )
                else:
                    res = ""
            except Exception as e:
                logging.warning(f"[Tool call failed] {name} -> {e}")
                res = ""
            out.append({"role": "tool", "content": str(res), "tool_call_id": tc.id})
        return out

    # Main planning loop
    def plan(self, memory=None):
        self.memory = memory or []
        self.opportunity = None

        msgs = [
            {"role": "system", "content": "Find deals, estimate value, notify user of best deal once."},
            {"role": "user", "content": "Scan, then estimate each, then notify for the best one. Reply OK when done."}
        ]

        while True:
            try:
                r = self.openai.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=msgs,
                    tools=self.get_tools()
                )
            except Exception as e:
                logging.warning(f"[Planner LLM call failed] {e}")
                break

            choice = r.choices[0]
            if choice.finish_reason != "tool_calls":
                break

            msgs.append(choice.message)
            msgs.extend(self.handle_tool(choice.message))

        return self.opportunity

In [ ]:
autonomous = AutonomousPlannerLLM()
result = autonomous.plan([])
print(result)

In [ ]:
app_fw = DealAgentFramework()
app_fw.planner = PlanningAgentLLM(app_fw.collection)

def table(opps):
    return [[o.deal.product_description, o.deal.price, o.estimate, o.discount, o.deal.url] for o in (opps or [])]

with gr.Blocks(title="Price is Right", fill_width=True) as ui:
    state = gr.State(app_fw.memory or [])
    gr.Markdown("**The Price is Right** – With OpenAI RAG, Anthropic, Google and OpenRouter")
    df = gr.Dataframe(headers=["Description", "Price", "Estimate", "Discount", "URL"], wrap=True, max_height=400)
    ui.load(table, state, df)
    def run():
        opps = app_fw.run()
        return table(opps), opps
    gr.Button("Run process").click(run, outputs=[df, state])
ui.launch(inbrowser=True)